In [1]:
import numpy as np
import gymnasium as gym
import matplotlib.pyplot as plt


In [2]:
def build_env(slippery):
  env = gym.make("FrozenLake-v1",is_slippery=slippery,render_mode = "rgb_array",map_name = "8x8")
  env.reset()
  return env

In [3]:
def get_transition_reward_matrices(env):
	n_state = env.observation_space.n
	n_action = env.action_space.n
	T = np.zeros((n_state, n_action, n_state), dtype=float)
	R = np.zeros((n_state, n_action), dtype=float)

	for S in range(n_state):
		for A in range(n_action):
			for prob, s , r, _terminated in env.unwrapped.P[S][A]: #s is the next state
				T[S, A, s] += prob
				R[S, A] += (prob*r)
	return T, R

In [4]:
def value_iteration(env, gamma, theta=1e-6):
  V = np.zeros(env.observation_space.n)
  T , R = get_transition_reward_matrices(env)
  iteration = 0
  while True:
    delta = 0
    for s in range(env.observation_space.n):
      v_old= V[s]
      Q = np.sum(T[s] * (R[s][:, None] + gamma * V), axis=1)
      v[s] = np.max(Q)
      delta = max(delta, abs(v_old - V[s]))
    iteration +=1
    if delta < theta:
      break
  policy = np.zeros(env.observation_space.n, dtype=int)
  for s in range(env.observation_space.n):
    Q = np.sum(T[s] * (R[s][:, None] + gamma * V), axis=1)
    policy[s] = np.argmax(Q)
  return V , policy , iteration


In [5]:
def policy_iteration(env , gamma , theta:1e-6):
  N_state = env.observation_space.n
  T , R = get_transition_reward_matrices(env)
  iteration = 0
  V = np.zeros(N_state)
  policy = np.zeros(N_state, dtype = int)
  while True:
    while True:
      delta = 0
      for s in N_state:
        V_old = V[s]
        V[s]  = np.sum(T[s, policy(s)] * (R[s, policy(s)] + gamma * V))
        delta = max(delta, abs(V_old - V[s]))
      if delta < theta:
        break
    policy_old = policy.copy()
    for s in range(N_state):
      Q = np.sum(T[s] * (R[s][:, None] + gamma * V), axis=1)
      policy[s] = np.argmax(Q)
    iteration += 1
    if np.array_equal(policy_old, policy):
      break
  return V , policy , iteration


In [6]:
def matrix_method(env, T, R, gamma=0.9):
  n_states = env.observation_space.n
  T_pi = np.zero((n_states, n_states))
  R_pi = np.zero(n_states)
  for s in range(n_states):
    T_pi[s] = T[s , policy[s] , :]
    R_pi[s] = R[s , policy[s]]
  I = np.identity(n_states)
  V = np.linalg.solve(I - gamma * T_pi , R_pi)
  return V

In [7]:
def evaluate_policy(env , policy , n_episodes = 1000):
  length = []
  Rewards = []
  for episode in range(n_episodes):
    rewards = 0
    length = 0
    s , info = env.reset()
    while True:
      a = policy[s]
      next_s , r , terminated , truncated  = env.step(a)
      rewards += r
      length += 1
      s = next_s
      if terminated or truncated:
        break
    Rewards.append(rewards)
    length.append(length)
  return np.mean(Rewards), np.mean(length)

In [8]:
def plot_policy_on_frozen_lake(env, policy, title="FrozenLake policy"):
	desc = np.asarray(env.unwrapped.desc, dtype=str)
	policy_grid = np.asarray(policy).reshape(desc.shape)
	arrows = np.array(["<", "v", ">", "^"])
	colors = {
		"S": "#9be7a1",
		"F": "#dceefb",
		"H": "#3a3a3a",
		"G": "#ffd54f",
	}

	fig, ax = plt.subplots(figsize=(8, 8))
	for r in range(desc.shape[0]):
		for c in range(desc.shape[1]):
			tile = desc[r, c]
			rect = plt.Rectangle((c, desc.shape[0] - 1 - r), 1, 1, facecolor=colors[tile], edgecolor="black", linewidth=1.5)
			ax.add_patch(rect)

			if tile == "H":
				label = "H"
			elif tile == "G":
				label = "G"
			elif tile == "S":
				label = f"S{arrows[policy_grid[r, c]]}"
			else:
				label = arrows[policy_grid[r, c]]

			ax.text(c + 0.5, desc.shape[0] - 1 - r + 0.5, label, ha="center", va="center", fontsize=16, fontweight="bold", color="black")

	ax.set_xlim(0, desc.shape[1])
	ax.set_ylim(0, desc.shape[0])
	ax.set_xticks(np.arange(desc.shape[1] + 1))
	ax.set_yticks(np.arange(desc.shape[0] + 1))
	ax.grid(True, color="black", linewidth=1.0)
	ax.set_xticklabels([])
	ax.set_yticklabels([])
	ax.set_aspect("equal")
	ax.set_title(title)
	plt.tight_layout()
	plt.show()
